In [ ]:
!pip install -q transformers accelerate torch pandas tqdm


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = "/content"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16
).cuda()

model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
df = pd.read_csv("new_data.csv")

print("Original distribution:")
print(df["label"].value_counts())

Original distribution:
label
0    369015
1    360561
Name: count, dtype: int64


In [ ]:
normal_df = df[df["label"] == 0].copy()
toxic_df = df[df["label"] == 1].copy()

# Safety
normal_df["label"] = 0
toxic_df = toxic_df.reset_index(drop=True)

print("\nNormal samples:", len(normal_df))
print("Toxic samples:", len(toxic_df))


Normal samples: 369015
Toxic samples: 360561


In [ ]:
# ---------------- CONFIG ----------------
BATCH_SIZE = 64
HATE_PERCENTILE = 90   # top 10% → hate
DEVICE = "cuda"
# ---------------------------------------

texts = toxic_df["content"].tolist()

loader = DataLoader(
    texts,
    batch_size=BATCH_SIZE,
    shuffle=False
)

hate_probs = []

model.eval()
with torch.no_grad():
    for batch_texts in tqdm(loader, desc="Running inference"):
        inputs = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

        # collect ONLY hate probability (class index 2)
        hate_probs.extend(probs[:, 2].cpu().numpy())

hate_probs = np.asarray(hate_probs, dtype=np.float64)

# Replace NaN / inf safely
hate_probs = np.nan_to_num(
    hate_probs,
    nan=0.0,
    posinf=1.0,
    neginf=0.0
)

Running inference: 100%|██████████| 5634/5634 [04:57<00:00, 18.92it/s]


In [ ]:
print("NaN count in hate_probs:", np.isnan(hate_probs).sum())
print("Total samples:", len(hate_probs))
print("Sample hate_probs:", hate_probs[:10])

NaN count in hate_probs: 0
Total samples: 360561
Sample hate_probs: [1.18827820e-03 9.89257812e-01 2.06146240e-02 2.57873535e-02
 8.07762146e-04 9.48242188e-01 2.39944458e-03 1.89495087e-03
 8.46862793e-04 1.81961060e-03]


In [ ]:
hate_probs = np.array(hate_probs)

hate_cutoff = np.percentile(hate_probs, HATE_PERCENTILE)

final_labels = np.where(
    hate_probs >= hate_cutoff,
    2,  # hate
    1   # offensive
)

toxic_df["label"] = final_labels
toxic_df["hate_score"] = hate_probs

In [ ]:
print("\nPost-inference toxic distribution:")
print(toxic_df["label"].value_counts())

assert 2 in toxic_df["label"].unique(), "❌ HATE LABEL MISSING — inference failed"


Post-inference toxic distribution:
label
1    324501
2     36060
Name: count, dtype: int64


In [ ]:
toxic_df_labeled = toxic_df.copy()
del toxic_df   # prevent accidental reuse

In [ ]:
final_df = pd.concat(
    [normal_df, toxic_df_labeled],
    ignore_index=True
)

final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print("\nFINAL DATASET DISTRIBUTION:")
print(final_df["label"].value_counts())
print(final_df["label"].value_counts(normalize=True))

print("\nUnique labels:", final_df["label"].unique())
assert set(final_df["label"].unique()) == {0, 1, 2}, "❌ Missing class in final data"


FINAL DATASET DISTRIBUTION:
label
0    369015
1    324501
2     36060
Name: count, dtype: int64
label
0    0.505794
1    0.444780
2    0.049426
Name: proportion, dtype: float64

Unique labels: [1 0 2]


In [ ]:
final_df.to_csv("new_data_processed.csv", index=False)
print("\n✅ Saved as new_data_processed.csv")


✅ Saved as new_data_processed.csv


In [ ]:
import pandas as pd

# ---------------- LOAD FILES ----------------
df_new = pd.read_csv("new_data_processed.csv")
df_all = pd.read_csv("all_data.csv")

print("Loaded:")
print("new_data_processed:", df_new.shape)
print("all_data:", df_all.shape)

Loaded:
new_data_processed: (729576, 3)
all_data: (44892, 2)


In [ ]:
def normalize(df):
    df = df.rename(columns={
        "Content": "content",
        "text": "content",
        "Text": "content",
        "Label": "label"
    })
    return df[["content", "label"]]

df_new = normalize(df_new)
df_all = normalize(df_all)

In [ ]:
df_new["label"] = df_new["label"].astype(int)
df_all["label"] = df_all["label"].astype(int)

print("\nLabel sets:")
print("new_data_processed:", sorted(df_new["label"].unique()))
print("all_data:", sorted(df_all["label"].unique()))


Label sets:
new_data_processed: [np.int64(0), np.int64(1), np.int64(2)]
all_data: [np.int64(0), np.int64(1), np.int64(2)]


In [ ]:
final_df = pd.concat(
    [df_new, df_all],
    ignore_index=True
)

In [ ]:
before = len(final_df)
final_df = final_df.drop_duplicates(subset=["content"])
after = len(final_df)

print(f"\nRemoved {before - after} duplicate rows")


Removed 150 duplicate rows


In [ ]:
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print("\nFINAL DATASET DISTRIBUTION:")
print(final_df["label"].value_counts())
print(final_df["label"].value_counts(normalize=True))

assert set(final_df["label"].unique()).issubset({0,1,2})


FINAL DATASET DISTRIBUTION:
label
0    380942
1    349964
2     43412
Name: count, dtype: int64
label
0    0.491971
1    0.451964
2    0.056065
Name: proportion, dtype: float64


In [ ]:
final_df.to_csv("final_data.csv", index=False)
print("\n✅ Saved as final_data.csv")


✅ Saved as final_data.csv
